# Experiment 5: System Characterization

Comparing 4 distinct operating points simultaneously.

In [ ]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import numpy as np
import matplotlib.pyplot as plt
from phantom import generate_qa_phantom
from forward_projection import generate_sinogram
from noise_model import add_poisson_noise
from fbp import reconstruct_fbp
from mtf import compute_mtf_from_reconstruction
from nps import compute_nps
from cnr import compute_cnr_from_qa_phantom
from metrics import compute_rmse
from utils import display_images_grid, plot_mtf_curves, plot_nps_curves

## 1. Setup Operating Points

In [ ]:
phantom, metadata = generate_qa_phantom(256)
# 1. High Dose, High Angle, Sharp (Baseline)
sino_360, ang_360 = generate_sinogram(phantom, 360)
noisy_highdose, _ = add_poisson_noise(sino_360, 1e6)
recon_1 = reconstruct_fbp(noisy_highdose, ang_360, "ram-lak")

# 2. High Dose, Low Angle, Sharp
sino_90, ang_90 = generate_sinogram(phantom, 90)
noisy_90, _ = add_poisson_noise(sino_90, 1e6)
recon_2 = reconstruct_fbp(noisy_90, ang_90, "ram-lak")

# 3. Low Dose, High Angle, Sharp
noisy_lowdose, _ = add_poisson_noise(sino_360, 1e4)
recon_3 = reconstruct_fbp(noisy_lowdose, ang_360, "ram-lak")

# 4. High Dose, High Angle, Smooth
recon_4 = reconstruct_fbp(noisy_highdose, ang_360, "hamming")

recons = [recon_1, recon_2, recon_3, recon_4]
titles = ["1. Baseline (1e6, 360, RamLak)", "2. Low Angles (1e6, 90, RamLak)", "3. Low Dose (1e4, 360, RamLak)", "4. Smooth (1e6, 360, Hamming)"]

## 2. Visual Comparison

In [ ]:
display_images_grid(recons, titles=titles, ncols=2, save_path="../results/exp5_visual.png")

## 3. Extract Metrics

In [ ]:
mtf_res, nps_res = {}, {}
for r, t in zip(recons, titles):
    f_mtf, m_vals = compute_mtf_from_reconstruction(r, method="edge")
    mtf_res[t] = (f_mtf, m_vals)
    f_nps, n_vals, _, _ = compute_nps(r, roi_size=32, num_rois=8, center=(128,128))
    nps_res[t] = (f_nps, n_vals)
    cnr, _ = compute_cnr_from_qa_phantom(r, metadata)
    rmse = compute_rmse(r, phantom)
    print(f"{t}: CNR={cnr:.2f}, RMSE={rmse:.4f}")

## 4. Plot MTF and NPS

In [ ]:
plot_mtf_curves(mtf_res, save_path="../results/exp5_mtf.png")
plot_nps_curves(nps_res, log_scale=True, save_path="../results/exp5_nps.png")